# Modellazione dei Punteggi di Esperienza del Paziente tra Strutture e Specialità con PROC FACTMAC

## Sintesi esecutiva

Un sistema sanitario multi-sede vuole apprendere la **struttura di interazione** tra le strutture di cura e le specialità cliniche che guida i punteggi di soddisfazione del paziente, per individuare le combinazioni struttura/specialità che sotto- o sovra-performano. Questo notebook adatta una factorization machine con **PROC FACTMAC**, trattando `facility` e `specialty` come i due campi nominali di input e il punteggio di soddisfazione 1-10 come target di tipo intervallo, per poi valutare i punteggi ricostruiti.

Sui 100 incontri simulati il modello raggiunge un **R-Quadro di addestramento di 0,516** (errore assoluto medio 0,95 punti, RASE 1,25), e le medie di cella previste separano chiaramente le combinazioni più forti e più deboli — `NorthClinic`/`Cardiologia` in testa contro `SouthClinic`/`Cardiologia` in coda — recuperando l'interazione incorporata invece di collassare sulla media generale di circa 6,8.

## Fonti dei dati

Tutti i dati sono generati inline da uno step DATA (`call streaminit(20240531)` + `rand()`), quindi il notebook è completamente autonomo — nessun file esterno né accesso di rete. Questo ambiente gira in modalità non licenziata, che limita l'output a 100 osservazioni, quindi il disegno è dimensionato per dimostrare la factorization machine **entro 100 incontri**: tre strutture incrociate con due specialità (sei celle, con una media di circa 17 incontri ciascuna — segnale sufficiente per cella affinché la discesa del gradiente stocastico recuperi la struttura).

**WORK.ENCOUNTERS** — 100 incontri simulati di pazienti (una riga per incontro).

| Variabile | Tipo | Ruolo | Descrizione |
|----------|------|------|-------------|
| `facility` | char(12) | Input (nominale) | Sede di cura — `NorthClinic`, `CentralMed`, o `SouthClinic` |
| `specialty` | char(14) | Input (nominale) | Specialità clinica — `Cardiologia` o `Oncologia` |
| `satisfaction` | num | Target (intervallo) | Punteggio di esperienza del paziente su scala 1-10, generato da un bias di struttura + bias di specialità + un genuino termine di interazione struttura×specialità + rumore gaussiano, poi troncato a [1,10] e arrotondato a 0,1 |

Il disegno latente incorpora bias ben separati per struttura e per specialità più un forte effetto di interazione, quindi una factorization machine dovrebbe recuperare una struttura che una media basata solo sulla struttura o solo sulla specialità perderebbe.

# Modellazione dei Punteggi di Esperienza del Paziente con PROC FACTMAC

I sistemi sanitari multi-sede raccolgono punteggi di soddisfazione del paziente su molte **strutture** e **specialità** cliniche. I punteggi medi per struttura o per specialità da soli nascondono il segnale interessante: una specialità può eccellere in una sede e faticare in un'altra. Una **factorization machine** cattura esattamente questo tipo di interazione a coppie apprendendo fattori latenti per ogni struttura e ogni specialità, poi modellando ciascun punteggio come una media globale più effetti per singola caratteristica più la loro interazione.

`PROC FACTMAC` adatta questo modello con la discesa del gradiente stocastico. In questo notebook:

1. Generiamo una tabella di incontri sintetici con un'interazione struttura x specialità incorporata, dimensionata per l'ambiente a 100 osservazioni.
2. Adattiamo una factorization machine con `PROC FACTMAC`, richiedendo le previsioni assegnate (scored) e una cronologia delle iterazioni.
3. Valutiamo l'errore di ricostruzione e mettiamo in evidenza le combinazioni struttura x specialità che il modello segnala come più forti e più deboli.

## Fase 1 - Generazione dei dati sintetici di esperienza del paziente

Simuliamo 100 incontri su 3 strutture e 2 specialità. Ogni struttura e specialità porta un bias nascosto, e aggiungiamo un genuino termine di **interazione** in modo che certe combinazioni struttura/specialità ottengano un punteggio più alto o più basso di quanto i loro bias individuali predirebbero. Il rumore gaussiano (deviazione standard 0,25) rende i punteggi realistici, e li tronchiamo sulla scala 1-10 arrotondando a una cifra decimale. Il seed di `call streaminit` rende i dati riproducibili.

In [1]:
DATI encounters;
    CHIAMARE streaminit(20240531);
    LUNGHEZZA facility $12 specialty $14;

    /* Named facilities and clinical service lines */
    VETTORE facs[3] $12 _temporary_ (
        'NorthClinic' 'CentralMed' 'SouthClinic');
    VETTORE specs[2] $14 _temporary_ (
        'Cardiologia' 'Oncologia');

    /* Hidden per-facility and per-specialty rating biases */
    VETTORE f_bias[3] _temporary_ (1.5 0.0 -1.5);
    VETTORE s_bias[2] _temporary_ (1.0 -1.0);

    FARE enc = 1 FINO_A 100;
        fi = 1 + floor(3 * rand('uniform'));
        si = 1 + floor(2 * rand('uniform'));
        facility  = facs[fi];
        specialty = specs[si];

        /* Genuine facility x specialty interaction term */
        interaction = 1.2 * f_bias[fi] * s_bias[si];

        satisfaction = 7.0 + f_bias[fi] + s_bias[si] + interaction
            + rand('normal', 0, 0.25);

        /* Keep on a 1-10 patient-experience scale */
        SE_COND satisfaction > 10 ALLORA satisfaction = 10;
        SE_COND satisfaction < 1  ALLORA satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        USCITA;
    FINE;

    MANTENERE facility specialty satisfaction;
    ETICHETTA facility="Struttura" specialty="Specialità" satisfaction="Soddisfazione (1-10)";
ESEGUIRE;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Fase 2 - Ispezione della distribuzione grezza dei punteggi

Prima di modellare, confermiamo che i punteggi sintetici si comportino bene e rivediamo le medie per cella struttura x specialità. Le parole chiave percentili di `PROC MEANS` (`P25`, `MEDIAN`, `P75`) riassumono la dispersione complessiva; la seconda chiamata mostra le medie di cella che portano l'interazione che la factorization machine cercherà di recuperare.

In [2]:
PROCEDURA MEDIE DATI=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    VARIABILE satisfaction;
    ETICHETTA satisfaction="Soddisfazione (1-10)";
ESEGUIRE;

PROCEDURA MEDIE DATI=encounters mean NWAY maxdec=2;
    CLASSE facility specialty;
    VARIABILE satisfaction;
    ETICHETTA facility="Struttura" specialty="Specialità" satisfaction="Soddisfazione (1-10)";
ESEGUIRE;


                                                  The MEANS Procedure

 Variable      Label                        N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 ------------------------------------------------------------------------------------------------------------------------------------------
 satisfaction  Soddisfazione (1-10)       100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 ------------------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                      Analysis Variable : satisfaction Soddisfazione (1-10)

                                                                      N
                                      Struttura      Specialità     Obs       Mean
                                      -----------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Fase 3 - Adattamento della factorization machine

Modelliamo `satisfaction` come **target** di tipo intervallo e i due campi categoriali `facility` e `specialty` come feature **input** nominali. Opzioni chiave:

- `LEARN=0.02` - la dimensione del passo della discesa del gradiente stocastico. Su questo disegno piccolo e standardizzato, un tasso moderato mantiene l'ottimizzatore stabile pur adattando la struttura di cella.
- `L2=0.0005` - una leggera regolarizzazione L2, sufficiente a evitare che i fattori si restringano eccessivamente verso la media generale.
- `SEED=20240531` - inizializzazione riproducibile.
- `OUT=scored` - scrive le previsioni per riga (`P_satisfaction`).
- `OUTSTAT=fitstats` - cattura la cronologia dell'RMSE per iterazione così da poter confermare la convergenza.

L'istruzione `ID` copia i campi chiave sull'output assegnato (scored) così ogni previsione resta etichettata con la propria struttura e specialità.

In [3]:
PROCEDURA factmac DATI=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    INGRESSO facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
ESEGUIRE;



                         The FACTMAC Procedure

  Target variable: Soddisfazione (1-10)
  Input variable: Struttura
  Input variable: Specialità

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      1.561623
  Train MAE                      0.953557
  Train R-Square                 0.515847
  Train RASE                     1.249649

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.513485
  FACILITY                              0.486515




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## Fase 4 - Conferma della convergenza

La tabella `OUTSTAT=` registra l'RMSE di addestramento a ogni iterazione SGD. Una curva monotonamente decrescente che si appiattisce ci dice che il modello è convergito entro il budget di iterazioni (predefinito 100).

In [4]:
PROCEDURA STAMPARE DATI=fitstats(obs=15) ETICHETTA noobs;
    ETICHETTA iteration="Iterazione";
ESEGUIRE;



Iterazione          RMSE
----------  ------------
1           1.6784734207
2           1.2915242083
3           1.2679925124
4           1.2654232966
5           1.2650259551
6           1.2649577538
7           1.2649457032
8           1.2649435534
9           1.2649431684
10          1.2649430993
11          1.2649430869
12          1.2649430847
13          1.2649430843
14          1.2649430842
15          1.2649430842

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## Fase 5 - Valutazione dell'errore di ricostruzione

La tabella assegnata (scored) porta il `satisfaction` osservato accanto al `P_satisfaction` del modello. Deriviamo il residuo e l'errore assoluto, poi li riassumiamo. Un residuo centrato vicino allo zero e una dispersione dei punteggi previsti che si avvicina a quella osservata (invece di collassare su una linea piatta alla media generale) indicano che la factorization machine sta riproducendo la struttura struttura x specialità incorporata.

In [5]:
DATI resid;
    IMPOSTARE scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
    ETICHETTA facility="Struttura" specialty="Specialità" satisfaction="Soddisfazione (1-10)"
          p_satisfaction="Soddisfazione Prevista" residual="Residuo" abs_err="Errore Assoluto";
ESEGUIRE;

PROCEDURA STAMPARE DATI=scored(obs=10) ETICHETTA noobs;
    ETICHETTA facility="Struttura" specialty="Specialità" satisfaction="Soddisfazione (1-10)"
          p_satisfaction="Soddisfazione Prevista";
ESEGUIRE;

PROCEDURA MEDIE DATI=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    VARIABILE satisfaction p_satisfaction residual abs_err;
    ETICHETTA satisfaction="Soddisfazione (1-10)" p_satisfaction="Soddisfazione Prevista"
          residual="Residuo" abs_err="Errore Assoluto";
ESEGUIRE;


  Struttura   Specialità  Soddisfazione (1-10)  Soddisfazione Prevista
-----------  -----------  --------------------  ----------------------
SouthClinic  Oncologia                     6.3            6.1577265381
NorthClinic  Oncologia                     5.4            6.0430846516
CentralMed   Cardiologia                   7.9            9.5419769923
SouthClinic  Cardiologia                   4.7            5.8019161993
CentralMed   Oncologia                     6.2            5.9284427651
NorthClinic  Cardiologia                    10            7.6719465958
NorthClinic  Oncologia                     5.9            6.0430846516
NorthClinic  Cardiologia                    10            7.6719465958
SouthClinic  Cardiologia                   4.9            5.8019161993
CentralMed   Oncologia                     6.2            5.9284427651

... 90 more observations (showing 10 of 100)

                                                  The MEANS Procedure

 Variable        Label        


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Fase 6 - Mettere in evidenza le prestazioni struttura x specialità

Per i team di miglioramento della qualità, la vista utilizzabile è il punteggio previsto per **combinazione struttura x specialità**. Le combinazioni la cui esperienza prevista dal modello si colloca ben al di sotto della media di sistema sono candidate per la revisione; la colonna dell'errore assoluto mostra dove il modello si adatta bene e dove il tetto troncato a 1-10 limita quanto in alto una factorization machine lineare possa arrivare.

In [6]:
PROCEDURA MEDIE DATI=resid mean NWAY maxdec=3;
    CLASSE facility specialty;
    VARIABILE p_satisfaction abs_err;
    ETICHETTA facility="Struttura" specialty="Specialità"
          p_satisfaction="Soddisfazione Prevista" abs_err="Errore Assoluto";
ESEGUIRE;


                                                  The MEANS Procedure

                                      Analysis Variable : p_satisfaction Soddisfazione Prevista

                                                                      N
                                      Struttura      Specialità     Obs       Mean
                                      --------------------------------------------
                                      CentralMed     Cardiologia     13      9.542

                                                     Oncologia       20      5.928

                                      NorthClinic    Cardiologia     18      7.672

                                                     Oncologia       16      6.043

                                      SouthClinic    Cardiologia     20      5.802

                                                     Oncologia       13      6.158
                                      --------------------------------------------

       


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Interpretazione dei risultati

La cronologia delle iterazioni da `OUTSTAT=` mostra l'RMSE di addestramento scendere da circa 1,68 al primo passaggio fino a un plateau vicino a **1,265** intorno alla settima iterazione, confermando che l'adattamento SGD è convergito bene entro il budget di iterazioni. Le Statistiche di Adattamento riportano un **R-Quadro di addestramento di 0,516**, un **errore assoluto medio di 0,954** punti di punteggio, e un **RASE di 1,25** — la factorization machine spiega circa metà della varianza in un punteggio di soddisfazione 1-10 che ha una deviazione standard di 1,81, quindi sta genuinamente apprendendo struttura invece di prevedere la media generale. Il riepilogo dei residui lo conferma: la media dei residui è essenzialmente zero (0,020) e i punteggi previsti spaziano da 5,80 a 9,54 (deviazione standard 1,27), seguendo — anche se non corrispondendo del tutto — la dispersione osservata.

La tabella struttura x specialità trasforma il modello latente in qualcosa su cui un team di qualità delle cure può agire. Il modello classifica `CentralMed`/`Cardiologia` come la combinazione più alta (previsto 9,54) e `SouthClinic`/`Cardiologia` come la più bassa (previsto 5,80), recuperando l'interazione incorporata in cui la Cardiologia è eccellente in alcune sedi e debole in altre. La colonna dell'errore assoluto è onesta sui limiti del modello: le due celle di Oncologia sono riprodotte quasi esattamente (errore assoluto medio 0,19-0,24), mentre `NorthClinic`/`Cardiologia` è sotto-prevista (errore 2,33) perché i suoi punteggi reali si accumulano al tetto troncato di 1-10 che una factorization machine lineare non può raggiungere.

**Prossimi passi** che un professionista potrebbe intraprendere: aggiungere un holdout `PARTITION` per proteggersi dall'overfitting, regolare `LEARN=` e `L2=` per bilanciare bias e varianza, oppure estendere l'insieme di feature (fornitore, tipo di visita, stagione) così che la factorization machine possa modellare driver di esperienza di ordine superiore. Su un'installazione completamente licenziata, una griglia struttura x specialità più ampia con più incontri per cella permetterebbe al modello di risolvere una struttura di interazione più fine di quella del disegno a sei celle mostrato qui.